# 05 - S3.4: the automatic pilot chain (Stage 3)

Runs the S3.4 chain of `docs/s34_spec.md`: docs wording -> pilot (switches, acceptance, recalibration ladder) -> RQ3
wiring check -> budget gate -> freeze -> EEM selection (D60) -> report. It takes many Colab sessions (about 10
GPU-hours, more if a switch or a ladder step fires); every session continues where the last one stopped.

**Every session:** Runtime -> Change runtime type -> **T4 GPU**; open this notebook from the commit-pinned link in the
chat; **Run all**. Keep the tab open. If the runtime is lost, reconnect (T4) and Run all again.

**The runtime ends by itself** after the bundle (`RELEASE_RUNTIME`); the bundle is also saved on Drive
(`Skripsi/stage3/s34/s34_bundle.json`) if the download is blocked.

**The chain stops by itself** only when a rule hands the decision to the user (a PAUSE line says why), when units
failed once (a RETRY line: the next session retries them in a fresh runtime), or when it is done. Either way the last
cell downloads `s34_bundle.json`: upload it to the chat.

**Settings (cell 0)** change only when the chat says so: `USER_DECISION` + `EXPECT_LOG` answer a pause; `REPIN` moves
the chain to a fixed commit (finished units are kept when they replay identically under the fix).

In [ ]:
# 0) Settings
NB_VERSION = "b18606b4c6fc"   # fingerprint of this notebook's cells; the setup cell refuses an outdated copy
USER_DECISION = None          # the chat's answer to a pause, e.g. {"answers": 7, "action": "accept"} (spec section 11)
EXPECT_LOG = None             # the last bundle's log, e.g. "8:3fa2c91b04de": refuses an older chain file on Drive
REPIN = False                 # or the commit the chat names after a code fix (D166; it must descend from the chain's)
RELEASE_RUNTIME = True        # end the Colab runtime after the bundle (D170; the bundle is also on Drive)
CHAIN_VERSION = 1             # fixed: a code fix is adopted by replay (REPIN), not a new chain version (D166)
CHAIN_FILE = "chain.json"

In [ ]:
# 1) Setup: timeline, GPU, Drive, pinned code (every runtime)
import os, sys, subprocess, tempfile, json, time, glob, shutil, hashlib
from pathlib import Path
T_FIRST_CELL = time.time()
REPO_URL = "https://github.com/notanut/eem-acc-skripsi.git"
IN_COLAB = "google.colab" in sys.modules
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "NONE")
if IN_COLAB and "T4" not in gpu.stdout:
    raise RuntimeError("No T4 GPU: Runtime -> Change runtime type -> T4 GPU (if none is offered, wait and retry).")
if IN_COLAB:
    from google.colab import drive
    t0 = time.time(); drive.mount("/content/drive"); T_MOUNT = time.time() - t0
    REPO_DIR = "/content/eem-acc-skripsi"
    DRIVE_ROOT = "/content/drive/MyDrive/Skripsi"
    def git(*args, cwd=REPO_DIR, capture=False):
        return subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=capture, text=True)
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        git("clone", REPO_URL, REPO_DIR, cwd=None)
    git("fetch", "origin")
    # the session's code is pinned to the commit its FIRST runtime used (recorded on Drive); later runtimes check it out
    sess_file = Path(DRIVE_ROOT) / "stage3" / "s34" / CHAIN_FILE
    pinned = json.loads(sess_file.read_text()).get("stamp", {}).get("commit") if sess_file.exists() else None
    target = pinned or git("rev-parse", "origin/main", capture=True).stdout.strip()
    if REPIN:                           # D166: the fixed commit the chat names; it must descend from the chain's commit
        target = git("rev-parse", REPIN, capture=True).stdout.strip()
        if pinned and subprocess.run(["git", "merge-base", "--is-ancestor", pinned, target], cwd=REPO_DIR).returncode:
            raise RuntimeError(f"REPIN {REPIN} does not descend from the chain's commit {pinned[:7]}: ask in the chat")
    head = git("rev-parse", "HEAD", capture=True).stdout.strip()
    if head != target:
        if "eem_acc" in sys.modules:
            raise RuntimeError("Other code is loaded: Runtime -> Restart session, then run from the top.")
        git("checkout", "--detach", target)
    dirty = git("status", "--porcelain", "--untracked-files=no", capture=True).stdout.strip()
    assert not dirty, f"the checkout is not clean: {dirty}"
    nb_src = "".join(json.loads((Path(REPO_DIR) / "notebooks" / "05_s34.ipynb").read_text())["cells"][1]["source"])
    if f'NB_VERSION = "{NB_VERSION}"' not in nb_src:
        raise RuntimeError(f"This notebook's cells differ from the S3.4 chain's pinned commit {target[:7]}: open "
                           f"the Colab link for commit {target[:7]} from the chat and run from the top.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt", "-e", REPO_DIR], check=True)
else:                                   # local test run: this checkout + a temp folder
    T_MOUNT = 0.0
    REPO_DIR = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
    DRIVE_ROOT = tempfile.mkdtemp(prefix="eem_acc_")
import importlib
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
importlib.invalidate_caches()
PINS_FILE = Path(REPO_DIR) / "configs" / "t4_pins.json"
PINS = json.loads(PINS_FILE.read_text()) if PINS_FILE.exists() else {}
from eem_acc.stage3.llm import t4                      # imports neither torch nor transformers
ROOT = Path(DRIVE_ROOT) / "stage3"                     # the campaign root (runner, epochs, ledger, lease, claims)
OUT = ROOT / "s34"; OUT.mkdir(parents=True, exist_ok=True)
REPO_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
S = t4.Sections(OUT / CHAIN_FILE, stamp={"commit": REPO_COMMIT, "nb_version": NB_VERSION}, repin=bool(REPIN),
                expect_log=EXPECT_LOG)   # pins the code; M6: before any write
if S.data.get("stamp", {}).get("commit") not in (None, REPO_COMMIT):
    raise RuntimeError(f"the S3.4 chain was started at another commit ({S.data['stamp']['commit']}); ask in the chat")
if IN_COLAB:                           # D142-D148: embedder libraries + MiniLM, BEFORE anything imports torch
    from eem_acc.stage3.llm import embed_setup as ES
    from eem_acc.stage3.llm.ledger import runtime_id
    def _sh(argv):
        r = subprocess.run(argv, capture_output=True, text=True)
        importlib.invalidate_caches()
        return r.returncode, r.stdout or "", r.stderr or ""
    def _freeze():
        rc, out, _ = _sh([sys.executable, "-m", "pip", "list", "--format=json", "--disable-pip-version-check"])
        return {d["name"]: d["version"] for d in json.loads(out.strip().splitlines()[-1])} if rc == 0 and out.strip() else None
    def _minilm(local_dir, rev):
        from huggingface_hub import snapshot_download
        snapshot_download(ES.MINILM_REPO, revision=rev, local_dir=str(local_dir), allow_patterns=ES.MINILM_FILES)
    LIBS = ES.ensure_embed(S, t4.Sections(OUT / "library_attempts.json"), PINS, Path(DRIVE_ROOT),
                           Path("/content"), _sh, _minilm, loaded=ES.loaded_libraries(), freeze=_freeze,
                           runtime=runtime_id(), log=print)
    print("embedder:", LIBS["mode"], LIBS["versions"], LIBS["durations"], "| probe equal to runtime 1:",
          S.data["embed_probes"][-1]["equal_first"], "| CPU:", LIBS["probe"]["cpu"])
from eem_acc.stage3 import feasibility as F
from eem_acc.stage3.llm import runner as R
from eem_acc.stage3.llm.ledger import vm_boot_time
RUNTIME_INFO = {"vm_boot": vm_boot_time(), "first_cell": T_FIRST_CELL, "mount_seconds": T_MOUNT, "commit": REPO_COMMIT,
                "gpu": gpu.stdout.strip()}
print("commit", REPO_COMMIT[:12], "| saved sections:", [k for k in S.data if k != "stamp"])

In [ ]:
# 2) Install llama-cpp-python with CUDA (pinned; the built wheel is cached on Drive)
import zipfile, zlib
LLAMA_VERSION = "0.3.36"           # released 2026-10-01 (PyPI); pinned for reproducibility
PYTAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
WHEEL_DIR = Path(DRIVE_ROOT) / "wheels"
WHEEL_DIR.mkdir(parents=True, exist_ok=True)
BUILD_DIR = Path(globals().get("BUILD_DIR_OVERRIDE", "/content/wheels"))   # override only in tests
def compatible(name):
    """llama-cpp-python loads its library through ctypes, so a source build is tagged py3-none-linux_x86_64;
    prebuilt wheels may carry the CPython tag instead. Accept both, never another Python version."""
    parts = name[:-4].split("-")          # dist, version, python tag, abi tag, platform
    return len(parts) == 5 and parts[2] in (PYTAG, "py3") and parts[4].startswith("linux_x86_64")
def good_wheels(folder=None):
    out = []
    for w in sorted(Path(folder or WHEEL_DIR).glob(f"llama_cpp_python-{LLAMA_VERSION}-*.whl")):
        if not compatible(w.name):
            continue
        try:
            with zipfile.ZipFile(w) as z:
                bad = z.testzip()
            if bad is None:
                out.append(w)
            else:
                print(f"ignoring a damaged wheel {w.name} (bad member {bad})")
        except (zipfile.BadZipFile, OSError, EOFError, zlib.error) as e:
            print(f"ignoring a broken wheel {w.name}: {e}")
    return out
def installed_version():
    r = subprocess.run([sys.executable, "-c", "import llama_cpp; print(llama_cpp.__version__)"], capture_output=True, text=True)
    return (r.stdout or "").strip() if r.returncode == 0 else None
def import_error():
    r = subprocess.run([sys.executable, "-c", "import llama_cpp"], capture_output=True, text=True)
    return (r.stderr or "").strip().splitlines()[-1:] if r.returncode != 0 else []
def save_to_drive(wheels):
    for w in wheels:
        tmp = WHEEL_DIR / (w.name + ".tmp")
        shutil.copy(w, tmp)
        os.replace(tmp, WHEEL_DIR / w.name)                 # never leave a half-copied wheel under the real name
    saved = good_wheels()
    if not saved:
        raise RuntimeError(f"copying the wheel to Drive failed; it is still in {BUILD_DIR}")
    return saved
already = installed_version() == LLAMA_VERSION
cached = good_wheels()
if IN_COLAB and not cached and good_wheels(BUILD_DIR):
    cached = save_to_drive(good_wheels(BUILD_DIR))          # an earlier build in this session that was not saved
    print(f"saved an earlier build from {BUILD_DIR} to Drive")
if IN_COLAB and not cached and not already:
    t0 = time.time()
    print("Building llama-cpp-python with CUDA for the T4 (sm_75). This takes about an hour the first time ...")
    env = dict(os.environ, CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75", CMAKE_BUILD_PARALLEL_LEVEL="4")
    subprocess.run([sys.executable, "-m", "pip", "wheel", "--no-deps", f"llama-cpp-python=={LLAMA_VERSION}",
                    "-w", str(BUILD_DIR)], env=env, check=True)
    built = good_wheels(BUILD_DIR)
    if not built:
        raise RuntimeError(f"build finished but no usable wheel in {BUILD_DIR}: {os.listdir(BUILD_DIR)}")
    cached = save_to_drive(built)
    print(f"built in {(time.time()-t0)/60:.1f} min")
print("cached on Drive:", [w.name for w in cached])
if cached and not already:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(cached[-1])], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "typing-extensions", "diskcache", "jinja2"], check=True)
if already and not cached:
    print("WARNING: llama-cpp-python is installed but no wheel is cached on Drive; wheel sha256 unknown")
# sha256 of the cached wheel (if the library was installed from elsewhere earlier in this session, it may differ)
WHEEL_SHA256 = hashlib.sha256(Path(cached[-1]).read_bytes()).hexdigest() if cached else None
print("llama-cpp-python:", installed_version(), "| wheel sha256:", WHEEL_SHA256)
if IN_COLAB and installed_version() != LLAMA_VERSION:
    hint = f" (delete {cached[-1]} on Drive to force a rebuild)" if cached else ""
    raise RuntimeError(f"llama-cpp-python {LLAMA_VERSION} is not importable: {import_error()}{hint}")

In [ ]:
# 3) Model: Drive cache (S3.0), local copy verified against S3.0's sha256 (every runtime)
from huggingface_hub import list_repo_files, hf_hub_download
MODEL_REPO, QUANT = "Qwen/Qwen2.5-7B-Instruct-GGUF", "q4_k_m"
MODEL_DIR = Path(DRIVE_ROOT) / "models" / MODEL_REPO.replace("/", "__")
files = sorted(f for f in list_repo_files(MODEL_REPO) if f.lower().endswith(".gguf") and QUANT in f.lower())
LOCAL = Path("/content/models") if IN_COLAB else Path(tempfile.mkdtemp())
LOCAL.mkdir(parents=True, exist_ok=True)
for f in files:
    src = Path(hf_hub_download(MODEL_REPO, f, local_dir=MODEL_DIR))
    dst = LOCAL / f
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy(src, str(dst) + ".tmp"); os.replace(str(dst) + ".tmp", dst)
MODEL_PATHS = [LOCAL / f for f in files]
PARTS = t4.verify_parts(MODEL_PATHS)
assert PARTS["equal_s30"], f"the model differs from S3.0's: {PARTS}"
PART_SHA = PARTS["sha256"]
print("model verified:", list(PART_SHA))

In [ ]:
# 4) Load the model on the GPU (every runtime)
import gc
from llama_cpp import Llama
for name in ("llm",):
    if globals().get(name) is not None:
        globals()[name].close(); globals()[name] = None
gc.collect()
t0 = time.time()
llm = Llama(model_path=str(MODEL_PATHS[0]), n_gpu_layers=-1, n_ctx=8192, n_batch=512, seed=0, verbose=False)
LOAD_SECONDS = time.time() - t0
from eem_acc.stage3.llm.llama_backend import load_params_of
from eem_acc.stage3.llm.canary import session_env
if not S.done("stamps"):
    S.save("stamps", {"wheel_sha256": WHEEL_SHA256, "wheel_equals_s30": WHEEL_SHA256 == t4.S30_WHEEL_SHA256,
                      "part_sha256": PART_SHA, "model_digest": t4.model_digest(PART_SHA),
                      "template_sha256": t4.template_check(llm)["sha256"], "load_params": load_params_of(llm),
                      "env": session_env(), "cuda_libraries": t4.cuda_libraries()})
print(f"loaded in {LOAD_SECONDS:.0f} s;", t4.gpu_telemetry(), "| wheel equals S3.0:", WHEEL_SHA256 == t4.S30_WHEEL_SHA256)

In [ ]:
# 6) Open the runtime: embedder, D146 kernel probe, CUDA check (recorded, D149), canary -> epoch (every runtime)
assert IN_COLAB, "S3.4 runs on the Colab T4 only"
assert os.path.ismount("/content/drive"), "Drive is not mounted (the session ended): Run all from the top"
assert WHEEL_SHA256, "no cached wheel sha256 (cell 2): ask in the chat"
STOP = None                                              # a reason to stop before the chain (the bundle still runs)
if globals().get("SES") is not None:                     # a re-run of this cell in the same kernel
    SES.close()
from eem_acc.stage3.llm.embed import MiniLMEmbedder, probe_vectors
from eem_acc.stage3.llm.ledger import runtime_id
EMB = MiniLMEmbedder(PINS["minilm_revision"], path=LIBS["minilm_path"])
assert EMB.cpu_capability == "AVX2" and EMB.id.endswith("isa-compat-avx2"), (EMB.cpu_capability, EMB.id)   # D143
EMB.embed("warm-up: torch, MKL and OpenMP initialised as during the run")
KP = t4.kernel_probe_check(probe_vectors(EMB), LIBS["probe"]["vectors_sha256"], runtime_id())
if not any(k["runtime"] == KP["runtime"] for k in S.data.get("kernel_probes", [])):
    S.append("kernel_probes", KP)
if not KP["equal"]:                      # D146 (strict): the in-process embedder must equal the setup's check
    STOP = "the kernel's embedder differs from the setup check (D146)"
s1 = json.loads((Path(REPO_DIR) / "docs/evidence/s33b_session1_2026-11-23.json").read_text())
prev = S.data.get("cuda_checks", [])
CC = t4.cuda_check(t4.cuda_libraries(), s1["stamps"]["cuda_libraries"], prev, runtime_id())
if not any(c["runtime"] == CC["runtime"] for c in prev):
    S.append("cuda_checks", CC)
if not CC["ok"]:                         # D149: recorded, not a stop; the canary decides the epoch
    print(">>> llama.cpp's CUDA libraries differ from session 1 or an earlier runtime (recorded; the canary decides) <<<")
if STOP is None:
    try:                                 # second opinion L2: a start-up failure stops (the session closes itself)
        SES = t4.T4Session(ROOT, llm, PART_SHA, WHEEL_SHA256, EMB, session_id="s34", load_seconds=LOAD_SECONDS)
    except Exception as e:
        import traceback
        traceback.print_exc()
        SES, STOP = None, f"the runtime did not start: {type(e).__name__}: {e}"
if STOP is None:
    RUN = SES.runner
    S.append("runtimes", {**RUNTIME_INFO, "runtime": SES.runtime, "epoch": SES.epoch, "canary": SES.status,
                          "fingerprint": SES.fingerprint, "broken_claims": SES.broken_claims,
                          "canary_seconds": SES.canary_seconds, "load_seconds": LOAD_SECONDS})
    print(f"runtime {len(S.data['runtimes'])}: epoch {SES.epoch} ({SES.status})")
    UD = USER_DECISION or {}
    if UD.get("action") == "allow_epoch" and UD.get("epoch") == SES.epoch and SES.epoch not in S.data.get(
            "allowed_epochs", []):           # the user's answer to an epoch pause (D165)
        S.append("allowed_epochs", SES.epoch)
    if SES.epoch != 0 and SES.epoch not in S.data.get("allowed_epochs", []):     # spec section 11.5: the user decides
        STOP = f"the canary put this runtime in epoch {SES.epoch}, not 0 (spec section 11.5)"
if STOP:
    print(f">>> PAUSE: {STOP}: the chain does not run; the bundle cell still runs - upload the bundle to the chat <<<")

In [ ]:
# 7) The S3.4 chain (re-run this cell to continue after an interruption in the same runtime)
from eem_acc.stage3 import s34 as S34
from eem_acc.stage3.s34_platform import Platform
from eem_acc.stage3.llm.cache import read_header
from eem_acc.stage3.llm.ledger import summarise, profile
from eem_acc.stage3.llm.llama_backend import key_source_from_header, LlamaTokenizer
VOCAB = Llama(model_path=str(MODEL_PATHS[0]), vocab_only=True, verbose=False)

def replay_tools(u):                                     # verify_replay's key source: the unit's cache header
    ks = key_source_from_header(read_header(RUN.cache_path(u)), VOCAB)
    return ks, LlamaTokenizer(ks)

assert not IN_COLAB or os.path.ismount("/content/drive"), "Drive is not mounted (the session ended): Run all"
CHAIN_STATE = None
if STOP:
    CHAIN_STATE = f"PAUSE: {STOP}"
else:
    PF = Platform(ROOT, RUN, REPO_COMMIT, replay_tools, version=CHAIN_VERSION, verified=S.data.get("verified"))
    CH = S34.Chain(S, ROOT, PF.run_units, PF.metrics, PF.checks, lambda: summarise(ROOT)["gpu_hours"],
                   lambda: profile(ROOT, "s34"), PF.freeze_extra, log=print, version=CHAIN_VERSION,
                   runtime=SES.runtime, adopt=PF.adopt)
    try:
        if USER_DECISION and USER_DECISION.get("action") != "allow_epoch":
            CH.apply_user_decision(USER_DECISION, EXPECT_LOG)
        CHAIN_STATE = CH.run()
    except S34.ChainRetry as e:                          # second opinion M3: no pause; the next session retries
        CHAIN_STATE = f"RETRY: {e}"
    except S34.ChainPause as e:
        CHAIN_STATE = f"PAUSE: {e}"
    except Exception as e:                               # never skip the bundle or leave the heartbeat on (review M5)
        import traceback
        traceback.print_exc()
        CHAIN_STATE = f"ERROR: {type(e).__name__}: {e}"
    except KeyboardInterrupt:                            # second opinion L2: an interrupt still writes the bundle
        CHAIN_STATE = "INTERRUPTED (the units already finished are kept; Run all continues the chain)"
print(">>>", CHAIN_STATE, "<<<")

In [ ]:
# 8) End of session: per-unit replay checks and audit, stop the heartbeat, write and download the bundle
import hashlib
from eem_acc.stage3 import s34 as S34
from eem_acc.stage3.llm.ledger import summarise
assert not IN_COLAB or os.path.ismount("/content/drive"), "Drive is not mounted (the session ended): Run all"
VERIFIED = dict(S.data.get("verified", {}))              # review M3: every finished unit, once, kept across sessions
try:
    if globals().get("PF") is not None:
        VERIFIED = PF.verify_finished(VERIFIED)
        S.save("verified", VERIFIED)
except Exception as e:                                   # the bundle is still written (fix review L1)
    import traceback
    traceback.print_exc()
    CHAIN_STATE = f"{globals().get('CHAIN_STATE')} | verify: {type(e).__name__}: {e}"
finally:
    if globals().get("SES") is not None:
        SES.close()
ff = ROOT / "s34" / S34.freeze_file(CHAIN_VERSION)
LOG_FP = S34.log_fingerprint(S.data.get("log", []))      # second opinion M6: the chat quotes it in EXPECT_LOG
bundle = {"chain": S.data, "freeze": json.loads(ff.read_text()) if ff.exists() else None,
          "ledger": {k: v for k, v in summarise(ROOT).items() if k != "unit_wall_seconds"},
          "state": globals().get("CHAIN_STATE"), "log_fingerprint": LOG_FP, "commit": REPO_COMMIT,
          "chain_sha256": hashlib.sha256((OUT / CHAIN_FILE).read_bytes()).hexdigest(),
          "not_verified": sorted(k for k, v in VERIFIED.items() if v.get("verify_replay") is not True),
          "audit_hits": sum(len(v.get("audit_hits", [])) for v in VERIFIED.values())}
BUNDLE = OUT / "s34_bundle.json"                          # the canonical copy, on Drive
BUNDLE.write_text(json.dumps(bundle, indent=1, default=str))
print("bundle:", BUNDLE, "| state:", bundle["state"], "| log:", LOG_FP, "| GPU-hours:", bundle["ledger"]["gpu_hours"])
if IN_COLAB:
    import shutil
    from google.colab import drive, files
    LOCAL_BUNDLE = shutil.copy(BUNDLE, "/content/s34_bundle.json")     # the download reads the VM's local copy
    FLUSHED = False
    try:
        drive.flush_and_unmount()                        # 2nd opinion #2, M4: block until Drive has every write
        FLUSHED = True
        print("Drive flushed: every write of this session is on Drive (cells 6-8 now need Run all from the top)")
    except Exception as e:                               # 2nd opinion #3, Low 4: keep the runtime so uploads finish
        print(f">>> Drive did not flush ({type(e).__name__}: {e}): the runtime is NOT ended; wait a few minutes, "
              "then end it yourself (Runtime -> Disconnect and delete runtime) <<<")
    files.download(LOCAL_BUNDLE)
if IN_COLAB and RELEASE_RUNTIME and FLUSHED:
    time.sleep(20)                                       # the browser fetches the download
    from google.colab import runtime
    runtime.unassign()